# Async servers: one callback handler per request

A server that compiles its agent once and reuses it across requests (cached per
user, say) can't use `AgentTraceMiddleware`: the middleware is part of the
compiled graph, so one instance would trace every request into the same run.
Attach a new `AsyncAgentTraceCallbackHandler` to each request instead, through
`config={"callbacks": [handler]}`. One handler traces the main agent and its
sub-agents: LLM calls (input, output, token usage), tool calls and results,
hand-offs to sub-agents, and errors.

The handler posts to a running `agenttrace ui` over HTTP. To stay offline, this
notebook runs that same server app in-process, on a SQLite file in a temporary
directory, and posts to it through httpx's ASGI transport: no port, no network,
and no API key (the server's `--auth none` mode).

```bash
pip install 'deepagents-trace[langchain,server]' jupyter
```

## Setup: an in-process AgentTrace server

In [ ]:
import os
import tempfile
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="agenttrace-notebook-"))
DB_URL = f"sqlite:///{workdir / 'agenttrace.db'}"
# What `agenttrace ui --database-url ... --auth none` sets up. The server reads
# DATABASE_URL when it is imported: restart the kernel to start over.
os.environ["DATABASE_URL"] = DB_URL
os.environ["AGENTTRACE_AUTH"] = "none"
os.environ.pop("AGENTTRACE_KEY", None)  # post without a key, whatever your shell exports

import httpx

import agenttrace
from agenttrace.langchain import AsyncAgentTraceCallbackHandler, AsyncHttpEventClient
from agenttrace.server.app import app  # the app `agenttrace ui` serves
from agenttrace.store.db import init_db

init_db(DB_URL)  # `agenttrace ui` creates the tables when it starts
server = httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://localhost")
client = AsyncHttpEventClient(url="http://localhost", client=server)  # one client, shared by all requests
agenttrace.set_project("notebook-async")  # without a key, a run names its project

In [ ]:
import json


async def show_last_run(payloads=False):
    """Print the latest run, read back through the REST API the dashboard uses."""
    latest = (await server.get("/api/runs/recent", params={"limit": 1})).json()["runs"][0]
    run = (await server.get(f"/api/runs/{latest['id']}")).json()["run"]
    print(f"run {run['name']!r} in project {run['project']['name']!r}: {run['status']}")
    for event in run["events"]:
        print(f"  {event['type']:<12} {event['source']:>14} → {event['target']:<14} {event['label'] or ''}")
        if payloads:
            print(f"  {'':12} payload {json.dumps(event['payload'], ensure_ascii=False)[:110]}")

## A cached agent with a sub-agent

The `support` agent hands order questions to an `orders` sub-agent through a
`task` tool, the way deepagents' `create_deep_agent(subagents=[...])` does; the
sub-agent looks the order up with a tool and reports back. Scripted chat models
stand in for the LLMs.

In [ ]:
from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langchain_core.tools import tool


class ScriptedChatModel(FakeMessagesListChatModel):
    model_name: str = "scripted-llm"

    def bind_tools(self, tools, **kwargs):
        return self


QUESTION = "Where is my order? I'm jane.doe@example.com"


@tool
def find_order(email: str) -> str:
    """Find the latest order placed with this email address."""
    return "Order #4521, shipped on 2026-10-10"


SUBAGENTS = {
    "orders": create_agent(
        ScriptedChatModel(
            responses=[
                AIMessage(content="", tool_calls=[{"name": "find_order", "args": {"email": "jane.doe@example.com"}, "id": "call_2"}]),
                AIMessage(content="Order #4521 shipped on 2026-10-10."),
            ]
        ),
        tools=[find_order],
        system_prompt="You look up orders.",
        name="orders",
    )
}


@tool
async def task(description: str, subagent_type: str) -> str:
    """Hand a task to a sub-agent and return its answer."""
    result = await SUBAGENTS[subagent_type].ainvoke({"messages": [{"role": "user", "content": description}]})
    return result["messages"][-1].content


# Compiled once, reused by every request.
agent = create_agent(
    ScriptedChatModel(
        responses=[
            AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "task",
                        "args": {"description": "Find the latest order of jane.doe@example.com", "subagent_type": "orders"},
                        "id": "call_1",
                    }
                ],
            ),
            AIMessage(content="Your order #4521 shipped on October 10."),
        ]
    ),
    tools=[task],
    system_prompt="You are a support agent. Hand order questions to the orders sub-agent.",
    name="support",
)

## One handler per request

This is what a request handler in your server does: create a handler, pass it in
the agent's `config`, and stream as usual. The handler sees the LLM and tool calls
on its own; the app supplies what a callback can't see: the user's message, the
answer it sends back, and the outcome. `aclose()` waits (bounded) until the
events are sent; a server would call it after sending its response.

In [ ]:
async def handle_request(question, **handler_options):
    handler = AsyncAgentTraceCallbackHandler(question, client=client, **handler_options)
    handler.on_user_message(question)
    answer, status = None, "completed"
    try:
        async for event in agent.astream_events(
            {"messages": [{"role": "user", "content": question}]},
            version="v2",
            config={"callbacks": [handler]},
        ):
            # Your own streaming to the client goes here; tracing needs nothing.
            if event["event"] == "on_chain_end" and not event["parent_ids"]:
                answer = event["data"]["output"]["messages"][-1].content
    except Exception as exc:
        status = "failed"
        handler.on_error(str(exc))
        raise
    finally:
        await handler.finish(status, answer=answer)
        await handler.aclose()
    return answer


print(await handle_request(QUESTION))
await show_last_run()

The `task` call is a hand-off arrow (Orchestrator → orders), and the sub-agent's
own LLM and tool calls are on the `orders` lane, not the orchestrator's: LangGraph
runs them under the `task` call's checkpoint namespace, which the handler
recognises. Only a tool named exactly `task`, or one whose name contains
`handoff` or `delegate`, counts as a hand-off.

Every request gets its own run, on the same compiled agent:

In [ ]:
await handle_request(QUESTION)
for run in (await server.get("/api/runs/recent")).json()["runs"]:
    print(run["id"][:8], run["status"], run["eventCount"], "events")

## Anonymizing what is sent

`anonymizer=` takes a function that the handler calls on every event payload (a
JSON-safe dict), every label and the run name (strings), so personal data is
masked before it leaves the process:

- it gets the full values: they are anonymized first and truncated afterwards,
  so a value cut at the size limit can't leak a fragment your patterns no longer
  match;
- if it raises, or returns something other than a string for a label or the run
  name, that field is redacted rather than sent unmasked, and one warning is
  logged;
- a sub-agent name read from the tool arguments is checked too: if the
  anonymizer changes it, or fails, the lane is called `Sub-agent`.

Participant names (agents, tools, models, `User`) are not anonymized.

In [ ]:
import re

EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")


def scrub(value):
    """Mask e-mail addresses in a payload (a dict of JSON values), a label or the run name."""
    if isinstance(value, str):
        return EMAIL.sub("<email>", value)
    if isinstance(value, dict):
        return {key: scrub(item) for key, item in value.items()}
    if isinstance(value, list):
        return [scrub(item) for item in value]
    return value

In [ ]:
await handle_request(QUESTION, anonymizer=scrub)
await show_last_run(payloads=True)

A failing anonymizer redacts instead:

In [ ]:
def broken(value):
    raise RuntimeError("the scrubber is down")


await handle_request(QUESTION, anonymizer=broken)
await show_last_run(payloads=True)

## Labels in your language

`phrases=` overrides the labels the handler writes, merged over
`DEFAULT_PHRASES`; `{name}` and `{target}` are filled in.

In [ ]:
from agenttrace.tracking.async_run import DEFAULT_PHRASES

print(DEFAULT_PHRASES)
await handle_request(
    QUESTION,
    phrases={
        "tool_result": "{name} → résultat",
        "delegate": "délégation → {target}",
        "subagent_done": "{name} → terminé",
        "final_answer": "réponse finale",
    },
)
await show_last_run()

## The lifecycle, and other options

| Call | When |
| --- | --- |
| `handler.on_user_message(text)` | the opening User → Orchestrator arrow |
| `handler.approval_required(info)` | a human-in-the-loop pause that your app detects (from the graph state) |
| `handler.on_error(message)` | a run-level failure to show on the diagram |
| `await handler.finish(status, answer=...)` | the answer your app assembled; the run ends `failed` for `"failed"` or `"error"`, else `completed` |
| `await handler.aclose()` | waits (bounded) for the queued events to be sent |

`tool_server=` takes a function of the tool name; its result is added to tool
arrows as `payload.server` (for example, which MCP server ran the tool).
`orchestrator=` renames the main agent's lane (default: `Orchestrator`).

## Against a real server

Start `agenttrace ui`, and build one client when your server starts:

```python
client = AsyncHttpEventClient(url="http://127.0.0.1:3000")  # posts to /api/events
client = AsyncHttpEventClient(url="https://traces.example.com", api_key="atr_...", client=my_httpx_async_client)
```

Or leave `client=` out: the handler then posts to the tracking URI
(`set_tracking_uri()` / `AGENTTRACE_TRACKING_URI`), which must be `http(s)://`,
with `AGENTTRACE_KEY` if it is set. A server running with `--auth password` needs
that key (a project key, `atr_...`); without a key, runs go to the current
project (`set_project()` / `AGENTTRACE_PROJECT`).

Tracing never breaks a request: events are sent from a background task, and the
first failure turns tracing off for that run, with one warning.

The runs above are in this notebook's database. To open them in the dashboard:

In [ ]:
print(f"agenttrace ui --database-url {DB_URL}")